# DFT analysis of monolayer and bilayer hBN

Ground-state (QuantumESPRESSO, PBE) calculations for the first benchmark of the project (`docs/04_benchmark_systems.md`):
bilayer hBN computed explicitly vs monolayer hBN with the second layer treated as an effective substrate
(Notes, Sec. 2). The **monolayer** is the reference both for the layer and for the "substrate"; the **bilayer**
(stackings AA' and AB) is the explicit reference.

Rule of this notebook: monolayer and bilayers share the **same** lattice constant, cell height, energy cutoff and
k grids, so that explicit and effective calculations differ only by the method (effect of the substrate, not of
the computational setup).

Choices (user, 2026-11-11):
- SG15 ONCV PBE scalar-relativistic pseudopotentials (`../pseudos`, version 1.2, no NLCC), no spin-orbit.
- Fixed in-plane lattice constant, the same for all systems: experimental value of bulk hBN, a = 2.504 Å (a PBE scan
  of the monolayer is kept only as a check).
- Fixed interlayer distance d = 3.33 Å (as in Guo et al., PRB 102, 205113 (2020)); PBE does not bind the layers.
  An optional DFT-D3 scan is a check only.
- Geometry along z: bilayer centred at z = 0 (layers at z = ±d/2). For the effective scheme ("layer" = top layer,
  "substrate" = bottom layer) the monolayer is also computed **in the exact position of each layer of each
  bilayer** (`ML_bot`, `ML_top_AA1`, `ML_top_AB`): chi0 of layer and substrate then come directly from Yambo, with
  no rotation/translation of the databases. How the yambo slab cutoff is centred with respect to the cell origin
  is still to be checked on the Lumen source before the GW step.

## How to run
- ismhpc only (driven by a Claude Code session or Jupyter on the frontend); runs go through slurm (`all12h`), with
  the QE environment loaded by `qe_module`. At most 2 nodes at a time for this project (user, 2026-11-11): the
  runs are 1-node jobs and each Dataset uses `num_tasks=2` (fewer if other jobs are running).
- Each section starts with the cell that defines its calculator and run folder: sections are independent.
- All runs are appended with `skip=True` (the default of the calculator): cells can be re-executed safely, runs
  with existing results are not recomputed.
- Run folder: `DFT_analysis/` (not tracked in git).

In [ ]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
from mppi import InputFiles as I, Calculators as C, Datasets as D, Parsers as P, Utilities as U
from mppi.Calculators import Tools
from mppi.Datasets import PostProcessing as PP
from mppi.Utilities import Constants as Const
from mppi.Utilities import LatticeUtils as LL
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# Calculator factory for ismhpc (32 cores per node; many MPI tasks are better for pw.x)
module = '/home/dalessandro/module_script/qe_module'

def qe_code(nodes=1, partition='all12h', time='11:59:00', ntasks_per_node=16, cpus_per_task=2, omp_num_threads=2):
    rr = C.RunRules(scheduler='slurm', partition=partition, memory='125000', time=time,
                    nodes=nodes, ntasks_per_node=ntasks_per_node, cpus_per_task=cpus_per_task,
                    omp_num_threads=omp_num_threads, pre_processing=module)
    return C.QeCalculator(rr, activate_BeeOND=True)

## Lattice and atomic positions

We use the hexagonal lattice of QuantumESPRESSO (`ibrav=4`):
$$
a_1 = a(1,0,0) \, ,\quad a_2 = a(-\tfrac{1}{2},\tfrac{\sqrt{3}}{2},0) \, ,\quad a_3 = a(0,0,\tfrac{c}{a})
$$
with $c$ the cell height (layer(s) + vacuum), $\mathrm{celldm(3)} = c/a$. Reciprocal vectors in units of $2\pi/a$:
$b_1 = (1,\tfrac{1}{\sqrt{3}},0)$, $b_2 = (0,\tfrac{2}{\sqrt{3}},0)$; high-symmetry points
$M = (0,\tfrac{1}{\sqrt{3}},0)$, $K = (\tfrac{1}{3},\tfrac{1}{\sqrt{3}},0)$.

The three in-plane sites of the honeycomb lattice (cartesian, units of $a$):
$$
S_A = (0,\tfrac{1}{\sqrt{3}}) \;[\text{crystal } (\tfrac13,\tfrac23)] \, ,\quad
S_B = (\tfrac12,\tfrac{1}{2\sqrt{3}}) \;[\text{crystal } (\tfrac23,\tfrac13)] \, ,\quad
S_C = (0,0) \;[\text{hexagon centre}]
$$
- Bottom layer (common to both bilayers): B on $S_A$, N on $S_B$ at $z=-d/2$.
- Bilayer AA' (stacking of bulk hBN): top layer N on $S_A$, B on $S_B$ at $z=+d/2$ (B over N and N over B). The top
  layer is the bottom one **rotated by 180°** about z; the bilayer has an inversion centre at the midpoint (D3d):
  the two layers are equivalent, no dipole.
- Bilayer AB (Bernal, parallel stacking): top layer B on $S_B$ (over the bottom N), N on $S_C$ (over a hexagon
  centre). The top layer is the bottom one **rigidly translated** in the plane; no inversion and no horizontal
  mirror (C3v): the layers are not equivalent and there is a small out-of-plane dipole (sliding ferroelectricity of
  parallel-stacked hBN). This is a ground-state interface effect NOT contained in the effective scheme. To check
  against the AB variant used by Guo et al. before the comparison.

Monolayers: `ML` (convergence studies) = `ML_bot` = the bottom layer alone; `ML_top_AA1` and `ML_top_AB` = the top
layer of each bilayer alone. Their DFT energies and bands are identical (same crystal, moved/rotated); they differ
in the plane-wave phases of the databases used later by Yambo.

In [ ]:
pseudo_dir = '../pseudos'
B_pseudo = 'B_ONCV_PBE-1.2.upf'
N_pseudo = 'N_ONCV_PBE-1.2.upf'

Bohr = 1./Const.Bohr_to_Angstrom   # bohr per angstrom (Constants gives the conversion factor bohr -> angstrom)

alat_ang = 2.504              # experimental in-plane lattice constant of bulk hBN (angstrom)
alat = alat_ang*Bohr          # in bohr (celldm(1))
d_ang = 3.33                  # interlayer distance (angstrom)
alat, alat_ang, d_ang

In [ ]:
SA = np.array([0., 1./np.sqrt(3.)])
SB = np.array([1./2., 1./(2*np.sqrt(3.))])
SC = np.array([0., 0.])

def hbn_positions(system, alat_ang=alat_ang, d_ang=d_ang):
    # Atomic positions in units of alat for the bilayers 'AA1' (AA'), 'AB' and the monolayers 'ML' (= 'ML_bot'),
    # 'ML_top_AA1', 'ML_top_AB' (each layer of the bilayers alone, in its own position)
    zt, zb = d_ang/2/alat_ang, -d_ang/2/alat_ang   # top and bottom layer heights (units of alat)
    def at(name, site, z): return [name, np.array([site[0], site[1], z])]
    layers = {'bot': [at('B', SA, zb), at('N', SB, zb)],
              'top_AA1': [at('N', SA, zt), at('B', SB, zt)],
              'top_AB': [at('B', SB, zt), at('N', SC, zt)]}
    if system in ['ML', 'ML_bot']: return layers['bot']
    if system == 'ML_top_AA1': return layers['top_AA1']
    if system == 'ML_top_AB':  return layers['top_AB']
    if system == 'AA1': return layers['bot'] + layers['top_AA1']
    if system == 'AB':  return layers['bot'] + layers['top_AB']
    raise ValueError('unknown system %s' % system)

systems = ['ML_bot', 'ML_top_AA1', 'ML_top_AB', 'AA1', 'AB']
for s in systems: print(s, hbn_positions(s))

Check of the geometry: positions in the xy and xz planes (a few cells).

In [ ]:
c_plot = 40.0  # bohr, only for this plot
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
for i, s in enumerate(['ML_top_AB', 'AA1', 'AB']):
    lattice = alat*np.array([[1,0,0],[-1./2.,np.sqrt(3)/2.,0],[0,0,c_plot/alat]])
    atoms = [[name, alat*pos] for name, pos in hbn_positions(s)]
    positions = LL.build_lattice(lattice, atoms, 4, 4, 1)
    for pos in positions:
        col = 'tab:red' if pos[0] == 'B' else 'tab:blue'
        ax[0, i].scatter(pos[1], pos[2], color=col, s=15)
        ax[1, i].scatter(pos[1], pos[3], color=col, s=15)
    ax[0, i].set_title('%s, xy (B red, N blue)' % s); ax[0, i].set_aspect('equal')
    ax[1, i].set_title('%s, xz' % s)
plt.tight_layout()

## Base input

`build_input(system, ...)` builds the scf input of a system with the given parameters: no spin-orbit, fixed
occupations (insulator). Number of bands: 4 occupied per layer, 4 empty per layer in the scf.

In [ ]:
nocc = {s: (8 if s in ['AA1', 'AB'] else 4) for s in ['ML'] + systems}   # occupied bands (8 valence electrons per layer)

def build_input(system, ecut, kpoints, c, alat=alat, conv_thr=1e-8, nbnd=None):
    # scf input; atomic positions in units of alat, computed for the fixed alat_ang (also when alat is scanned)
    inp = I.PwInput()
    inp.set_scf(force_symmorphic=True, conv_thr=conv_thr)
    inp.set_num_bnds(nbnd if nbnd is not None else 2*nocc[system])
    inp.set_pseudo_dir(pseudo_dir=pseudo_dir)
    inp.add_atom(atom='B', pseudo_name=B_pseudo, mass=10.811)
    inp.add_atom(atom='N', pseudo_name=N_pseudo, mass=14.007)
    pos = hbn_positions(system, alat_ang=alat/Bohr)   # z positions rescaled so that d stays 3.33 A
    inp.set_atoms_number(len(pos))
    inp.set_atomic_positions(pos, type='alat')
    inp.set_lattice(ibrav=4, celldm1=alat, celldm3=c/alat)
    inp.set_energy_cutoff(ecut)
    inp.set_kpoints(type='automatic', points=[kpoints, kpoints, 1])
    return inp

def gap_gamma(r, system):
    # gap at Gamma (first k point of the automatic grid)
    ev = r.get_evals()[0]
    return ev[nocc[system]] - ev[nocc[system]-1]

# starting guesses (to be replaced by the converged values below)
ecut_0 = 80     # Ry
k_0 = 12
c_0 = 40.0      # bohr (cell height)
build_input('AA1', ecut_0, k_0, c_0)

## Monolayer: convergence

### Energy cutoff and k grid
Control quantities: total energy, gap (`get_gap`: direct and indirect), gap at $\Gamma$. Grids multiple of 3 contain
$K$.

In [ ]:
run_dir = 'DFT_analysis'
code = qe_code()

In [ ]:
kvalues = [9, 12, 15, 18]
ecut = [50, 60, 70, 80, 90, 100]

study = D.Dataset(run_dir=run_dir, num_tasks=2)
for k in kvalues:
    for e in ecut:
        idd = {'system': 'ML', 'ecut': e, 'k': k}
        name = D.name_from_id(idd)
        inp = build_input('ML', e, k, c_0)
        inp.set_prefix(name)
        study.append_run(id=idd, input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_parse_data)
len(study.ids)

In [ ]:
results = study.run()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
for k in kvalues:
    data = study.fetch_results(id={'k': k})
    ax[0].plot(ecut, [d.energy for d in data], 'o-', label='k=%s' % k)
    ax[1].plot(ecut, [d.get_gap(verbose=False)['gap'] for d in data], 'o-', label='k=%s' % k)
    ax[2].plot(ecut, [gap_gamma(d, 'ML') for d in data], 'o-', label='k=%s' % k)
for a, yl in zip(ax, ['Total energy (Ha)', 'Gap (eV)', r'Gap at $\Gamma$ (eV)']):
    a.set_xlabel('energy cutoff (Ry)', size=14); a.set_ylabel(yl, size=14); a.legend()

On the basis of these results we choose

In [ ]:
ecut_conv = 80   # Ry, TO BE SET from the analysis
k_conv = 12      # TO BE SET

### Lattice constant (check only)
PBE equilibrium lattice constant of the monolayer, to know how far it is from the fixed experimental value. It is
NOT used in the following.

In [ ]:
alat_span = np.linspace(-2, 2, 9)   # percentage variation

study = D.Dataset(run_dir=run_dir, num_tasks=2)
for a_span in alat_span:
    idd = {'system': 'ML', 'alat_span': a_span}
    name = D.name_from_id(idd)
    inp = build_input('ML', ecut_conv, k_conv, c_0, alat=alat*(1+a_span/100))
    inp.set_prefix(name)
    study.append_run(id=idd, input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_get_energy)

In [ ]:
results = study.run()

In [ ]:
y = [results[ind] for ind in range(len(study.ids))]
x = [alat_ang*(1+a/100) for a in alat_span]
coeffs = np.polyfit(x, y, 2)
x_fit = np.linspace(x[0], x[-1], 50)
plt.plot(x, y, 'o'); plt.plot(x_fit, np.polyval(coeffs, x_fit))
plt.xlabel('a (angstrom)'); plt.ylabel('Total energy (Ha)')
alat_pbe = -coeffs[1]/(2*coeffs[0])
print('PBE equilibrium a = %.4f angstrom (fixed value %.4f)' % (alat_pbe, alat_ang))

### Cell height
The cell height $c$ must be the same for monolayer and bilayer, so it is converged on the bilayer (AA'), which has
the smaller vacuum ($c - d$), and checked on the monolayer. At the DFT level this fixes the spurious interaction
between replicas; with the slab cutoff in GW it will be re-examined there.

In [ ]:
c_values = [30, 35, 40, 45, 50]   # bohr

study = D.Dataset(run_dir=run_dir, num_tasks=2)
for s in ['ML', 'AA1']:
    for cval in c_values:
        idd = {'system': s, 'c': cval}
        name = D.name_from_id(idd)
        inp = build_input(s, ecut_conv, k_conv, cval)
        inp.set_prefix(name)
        study.append_run(id=idd, input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_parse_data)

In [ ]:
results = study.run()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
for s in ['ML', 'AA1']:
    data = study.fetch_results(id={'system': s})
    ax[0].plot(c_values, [d.energy for d in data], 'o-', label=s)
    ax[1].plot(c_values, [d.get_gap(verbose=False)['gap'] for d in data], 'o-', label=s)
    ax[2].plot(c_values, [gap_gamma(d, s) for d in data], 'o-', label=s)
for a, yl in zip(ax, ['Total energy (Ha)', 'Gap (eV)', r'Gap at $\Gamma$ (eV)']):
    a.set_xlabel('c (bohr)', size=14); a.set_ylabel(yl, size=14); a.legend()

On the basis of this analysis we choose

In [ ]:
c_conv = 40.0   # bohr, TO BE SET

## Bilayer: interlayer distance (optional check)
The production value is fixed, d = 3.33 Å. Here a scan with the DFT-D3 correction (`vdw_corr='dft-d3'`) for AA' and AB
gives the equilibrium distance as a check only.

In [ ]:
d_values = [3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6]   # angstrom

study = D.Dataset(run_dir=run_dir, num_tasks=2)
for s in ['AA1', 'AB']:
    for dv in d_values:
        idd = {'system': s, 'd': dv, 'vdw': 'd3'}
        name = D.name_from_id(idd)
        inp = build_input(s, ecut_conv, k_conv, c_conv)
        inp.set_atomic_positions(hbn_positions(s, d_ang=dv), type='alat')
        inp['system']['vdw_corr'] = "'dft-d3'"
        inp.set_prefix(name)
        study.append_run(id=idd, input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_get_energy)

In [ ]:
results = study.run()

In [ ]:
for s in ['AA1', 'AB']:
    y = study.fetch_results(id={'system': s})
    coeffs = np.polyfit(d_values, y, 4)
    x_fit = np.linspace(d_values[0], d_values[-1], 200)
    plt.plot(d_values, y, 'o', label=s); plt.plot(x_fit, np.polyval(coeffs, x_fit))
    print(s, 'D3 equilibrium d = %.3f angstrom' % x_fit[np.argmin(np.polyval(coeffs, x_fit))])
plt.xlabel('d (angstrom)'); plt.ylabel('Total energy (Ha)'); plt.legend()

## Converged ground states
Final scf of the three monolayers (bottom layer, top layer of AA', top layer of AB) and of the two bilayers with
the common converged parameters.

In [ ]:
run_dir = 'DFT_analysis'
code = qe_code()

ecut_conv = 80     # Ry, TO BE SET (same values as above)
k_conv = 12
c_conv = 40.0      # bohr

study = D.Dataset(run_dir=run_dir, num_tasks=2)
for s in systems:
    name = 'gs_converged-%s' % s
    inp = build_input(s, ecut_conv, k_conv, c_conv)
    inp.set_prefix(name)
    study.append_run(id=s, input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_parse_data)

In [ ]:
results = study.run()

In [ ]:
gs = {}
for ind, s in enumerate(systems):
    r = results[ind]; gs[s] = r
    g = r.get_gap(verbose=False)
    print(s, 'energy (Ha) = %.6f' % r.energy, '| gap = %.4f eV | direct gap = %.4f eV | gap at Gamma = %.4f eV'
          % (g['gap'], g['direct_gap'], gap_gamma(r, s)))

## Band structures on the path $\Gamma$-M-K-$\Gamma$
Bands of the three systems from the converged ground states. The comparison monolayer vs bilayer measures how much
the ground state of each layer changes in the bilayer (assumption of Notes, Sec. 2.1, on the KS states), and shows
the bonding/antibonding splitting of the degenerate states of the two layers. Bands are aligned at the VBM (an
alignment to the vacuum level can be added later).

In [ ]:
G = [0., 0., 0.]
Kp = [1./3., 1./np.sqrt(3), 0.]
Mp = [0., 1./np.sqrt(3), 0.]
high_sym = {'G': G, 'K': Kp, 'M': Mp}
klist = Tools.build_pw_kpath(G, Mp, Kp, G, numstep=60)

band_systems = ['ML_bot', 'AA1', 'AB']
nbnd_bands = {'ML_bot': 12, 'AA1': 24, 'AB': 24}
bands = {}
for s in band_systems:
    name = 'bands-%s' % s
    inp = build_input(s, ecut_conv, k_conv, c_conv)
    inp.set_bands(nbnd_bands[s], force_symmorphic=True, conv_thr=1e-8)
    inp.set_prefix(name)
    inp.set_kpoints(type='tpiba_b', klist=klist)
    res = code.run(input=inp, run_dir=run_dir, name=name, source_dir='%s/gs_converged-%s.save' % (run_dir, s))
    bands[s] = U.BandStructure.from_Pw(res, high_sym)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(20, 7), sharey=True)
for a, s in zip(ax, band_systems):
    bands[s].plot(plt, axes=a)
    a.set_title(s, size=16)
ax[0].set_ylabel('Band structure (eV)', size=14)

In [ ]:
# Overlay monolayer vs bilayers
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(16, 7), sharey=True)
for a, s in zip(ax, ['AA1', 'AB']):
    bands[s].plot(plt, axes=a, color='tab:red', label=s)
    bands['ML_bot'].plot(plt, axes=a, color='black', linestyle='--', label='ML')
    a.set_title('ML vs %s' % s, size=16)

## AB bilayer: dipole correction (check)
The AB bilayer has an out-of-plane dipole: in the periodic cell it produces a spurious field across the vacuum.
We repeat the AB scf with the dipole correction of pw.x (`tefield`, `dipfield`, sawtooth in the vacuum region,
`edir=3`; the layers sit at crystal z ~ ±0.08, the sawtooth at 0.5-0.6) and compare energy and gaps. The
correction is only a check: the yambo databases are built without it unless the effect is relevant.

In [ ]:
study = D.Dataset(run_dir=run_dir, num_tasks=1)
name = 'gs_converged-AB-dipfield'
inp = build_input('AB', ecut_conv, k_conv, c_conv)
inp['control']['tefield'] = '.true.'
inp['control']['dipfield'] = '.true.'
inp['system']['edir'] = 3
inp['system']['emaxpos'] = 0.5
inp['system']['eopreg'] = 0.1
inp['system']['eamp'] = 0.0
inp.set_prefix(name)
study.append_run(id='AB-dipfield', input=inp, runner=code, name=name)
study.set_postprocessing_function(PP.pw_parse_data)
results = study.run()

In [ ]:
r = results[0]
g, g0 = r.get_gap(verbose=False), gs['AB'].get_gap(verbose=False)
print('AB with dipole correction: energy %.6f Ha, gap %.4f eV, direct gap %.4f eV' % (r.energy, g['gap'], g['direct_gap']))
print('AB without               : energy %.6f Ha, gap %.4f eV, direct gap %.4f eV' % (gs['AB'].energy, g0['gap'], g0['direct_gap']))

## NSCF on regular grids (for Yambo)
To be set up with the GW analysis (k grids, number of bands, folders for the SAVE databases): same grids and bands
for monolayer and bilayers.